In [1]:
from pathlib import Path
from pydoc import describe

import nibabel as nib
import pandas as pd

from resources.find_centre import find_centre
from resources.find_pitch_error_y_y import find_pitch_error

from resources.cut_off_edges import cut_off_edges
from resources.find_rotation_error import find_rotation_error, find_rotation_error_by_profile
from resources.rotate_image import rotate_volume
from resources.find_tilt_error import find_tilt_error, find_tilt_error_by_profile
from resources.standarize_image import standardize_image

In [2]:
project_dir = Path.cwd().resolve().parents[2]
# print(project_dir)
niftis_dir = project_dir / "data" / "niftis"
outputs_dir = project_dir / "src" / "outputs" / "cut_images"

outputs_dir.mkdir(parents=True, exist_ok=True)

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"
labels = pd.read_csv(labels_path)


In [3]:
from notebooks.register_datscan_ants import register_to_template
from nibabel.processing import resample_to_output

results = []

for index, nifti_path in enumerate(nifti_paths[100:150], start=1):
    # Odczyt obrazu jako obiekt NIfTI
    try:
        image = register_to_template(nifti_path)
    except Exception as exc:
        print(f"Registration failed for {nifti_path}: {exc}", flush=True)
        image = nib.load(nifti_path)
    processed_image = cut_off_edges(image)

    processed_image = cut_off_edges(
        processed_image,
        x_left=35,
        x_right=35,
        y_front=40,
        y_back=40,
        z_down=30,
        z_up=30,
    )

    processed_image = standardize_image(
        processed_image,
        percent_top=0.02,
    )

    processed_image = resample_to_output(
        processed_image,
        voxel_sizes=(2.46, 2.46, 2.46),
        order=1,
    )

    rotation_error = find_rotation_error_by_profile(processed_image)
    tilt_error = find_tilt_error_by_profile(processed_image)
    pitch_error = None


    uid = Path(nifti_path).name.replace(".nii.gz", "")

    pathological = labels.loc[
        labels["uid"] == uid,
        "is_pathologic"
    ].iloc[0]

    results.append({
            "uid": uid,
            "pathological": pathological,
            "rotation_error": rotation_error,
            "tilt_error": tilt_error,
            "pitch_error": pitch_error
    })
    print(index, uid, rotation_error)
errors_df = pd.DataFrame(results)



1 2m71hssb -27
2 2movs0cd 5
3 2p2tixc7 6
4 2qc6dr08 -13
5 2r9pegln 6
6 2rntjlmz 5
7 2rzv1sqq -8
8 2u9az3ce -3
9 2xcb3yym 5
10 2xg8p6be 2
11 2xog4c8j 21
12 2ygnmby4 10
13 2z0ljch2 6
14 2zueugwu 2
15 313fpoyi -15
16 31eiwucx -4
17 31yhxso7 1
18 32orwixz -3
19 33mg1zaq 0
20 344mw04i 6
21 349n1d6d 3
22 356wafft 3
23 36u8ueih 11
24 39yl9ue4 3
25 3aipjiyl -2
26 3b0n9cc7 -2
27 3c9xqym5 -5
28 3cup8ijl 4
29 3dg60e1e 6
30 3dzxlrgn -1
31 3e54n6o9 -5
32 3fezez5e -4
33 3g5dfrlw 4
34 3hzcd3ge -7
35 3ifhv1xg 5
36 3iwpfvb7 0
37 3jczucqm 9
38 3jonou9i 22
39 3jpxgh30 -4
40 3k8a7rv4 -11
41 3kcvjxzd 4
42 3m9iqbq9 24
43 3obyghgk -13
44 3p91i3vm 9
45 3pt5r5rc 4
46 3qlx3f2x 1
47 3qsqcq30 16
48 3t4y0tb1 0
49 3uvulemx 4


KeyboardInterrupt: 

In [ ]:
errors_df.describe()

In [ ]:
errors_df[(errors_df["rotation_error"] == 12)]

In [ ]:
errors_df[(errors_df["tilt_error"] == -17)]

In [ ]:
# max_error = errors_df["rotation_error"].max()

errors_df[(errors_df["rotation_error"] == 0) & (errors_df["tilt_error"] == 1)]

In [ ]:
from matplotlib import pyplot as plt

plt.figure(figsize=(10, 5))

errors_df["rotation_error"].value_counts().sort_index().plot(
    kind="bar"
)

plt.xlabel("Rotation error [°]")
plt.ylabel("Liczba obrazów")
plt.title("Rozkład rotation error")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

errors_df["tilt_error"].value_counts().sort_index().plot(
    kind="bar"
)

plt.xlabel("Tilt error [°]")
plt.ylabel("Liczba obrazów")
plt.title("Rozkład tilt error")
plt.show()